In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

In [0]:
def multi_col_apply(func):
    def wrapper(df, cols=None):
        if cols is None:
            cols = df.columns
        for c in cols:
            df = func(df, c)
        return df

    return wrapper


def apply_transformations(df, transformations):
    for func, kwargs in transformations:
        df = func(df, **kwargs)
    return df


def save_unified_data(unify_func, offline_df, online_df, table_name, mode="append"):
    unified_df = unify_func(offline_df, online_df)
    unified_df.write.mode(mode).saveAsTable(table_name)

In [0]:
@multi_col_apply
def trimming(df, col):
    return df.withColumn(col, F.trim(F.col(col)))


@multi_col_apply
def uppercase(df, col):
    return df.withColumn(col, F.upper(F.col(col)))


@multi_col_apply
def lowercase(df, col):
    return df.withColumn(col, F.lower(F.col(col)))


@multi_col_apply
def capitalize(df, col):
    return df.withColumn(col, F.initcap(F.col(col)))


def mobile_num_standardize(df, col):
    new_col_name = col + "_cleaned"
    return df.withColumn(
        new_col_name, F.substring(col, F.length(col) - 9, 10)
    ).withColumn(
        f"valid_{col}",
        F.when(F.length(F.col(new_col_name)) == 10, True).otherwise(False),
    )


def handle_duplicate(df, col1, col2):
    return (
        df.withColumn(
            "rn",
            F.rank().over(
                Window.partitionBy(F.col(f"{col1}")).orderBy(F.col(f"{col2}"))
            ),
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

In [0]:
def unify_customers_data(df_offline_customers, df_online_customers):
    online_cust_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["full_name", "city"]}),
        (lowercase, {"cols": ["email"]}),
        (uppercase, {"cols": ["customer_id", "state"]}),
        (mobile_num_standardize, {"col": "mobile_number"}),
        (handle_duplicate, {"col1": "mobile_number_cleaned", "col2": "customer_id"}),
    ]

    df_online_transformed = apply_transformations(
        df_online_customers, online_cust_transformations
    )

    offline_cust_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["name", "city"]}),
        (lowercase, {"cols": ["email"]}),
        (uppercase, {"cols": ["cust_id", "state"]}),
        (mobile_num_standardize, {"col": "phone"}),
        (handle_duplicate, {"col1": "phone_cleaned", "col2": "cust_id"}),
    ]

    df_offline_transformed = apply_transformations(
        df_offline_customers, offline_cust_transformations
    )

    df_customers_unified = (
        df_online_transformed.alias("on")
        .join(
            df_offline_transformed.alias("off"),
            F.col("on.mobile_number_cleaned") == F.col("off.phone_cleaned"),
            "full_outer",
        )
        .select(
            F.coalesce(F.col("on.full_name"), F.col("off.name")).alias("full_name"),
            F.coalesce(F.col("on.email"), F.col("off.email")).alias("email"),
            F.coalesce(
                F.col("on.mobile_number_cleaned"), F.col("off.phone_cleaned")
            ).alias("mobile_number"),
            F.coalesce(F.col("on.city"), F.col("off.city")).alias("city"),
            F.coalesce(F.col("on.state"), F.col("off.state")).alias("state"),
            F.coalesce(F.col("on.zipcode"), F.col("off.pincode")).alias("zipcode"),
            F.col("on.signup_date").cast("date").alias("signup_date"),
            F.col("off.cust_id").alias("offline_customer_id"),
            F.col("on.customer_id").alias("online_customer_id"),
            F.coalesce(F.col("on.file_date"), F.col("off.file_date")).alias(
                "file_date"
            ),
            F.coalesce(F.col("on.source"), F.col("off.source")).alias("source"),
            F.coalesce(F.col("on.ingestion_ts"), F.col("off.ingestion_ts")).alias(
                "ingestion_ts"
            ),
        )
    )
    return df_customers_unified

In [0]:
def unify_products_data(df_offline_products, df_online_products):
    online_products_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["product_name", "category", "brand"]}),
        (uppercase, {"cols": ["sku_id"]}),
    ]

    df_online_transformed = apply_transformations(
        df_online_products, online_products_transformations
    )

    offline_products_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["name", "category_name", "brand_name"]}),
        (uppercase, {"cols": ["sku"]}),
    ]

    df_offline_transformed = apply_transformations(
        df_offline_products, offline_products_transformations
    )

    df_products_unified = (
        df_online_transformed.alias("on")
        .join(
            df_offline_transformed.alias("off"),
            F.col("on.sku_id") == F.col("off.sku"),
            "full_outer",
        )
        .select(
            F.coalesce(F.col("on.sku_id"), F.col("off.sku")).alias("product_id"),
            F.coalesce(F.col("on.product_name"), F.col("off.name")).alias(
                "product_name"
            ),
            F.coalesce(F.col("on.category"), F.col("off.category_name")).alias(
                "product_category"
            ),
            F.coalesce(F.col("on.brand"), F.col("off.brand_name")).alias(
                "product_brand"
            ),
            F.coalesce(F.col("on.price"), F.col("off.mrp_price"))
            .cast("double")
            .alias("product_price"),
            F.col("off.tax_rate").cast("float").alias("tax_rate"),
            F.col("on.currency").alias("currency"),
            F.col("on.is_active").alias("is_active"),
            F.coalesce(F.col("on.file_date"), F.col("off.file_date")).alias(
                "file_date"
            ),
            F.coalesce(F.col("on.source"), F.col("off.source")).alias("source"),
            F.coalesce(F.col("on.ingestion_ts"), F.col("off.ingestion_ts")).alias(
                "ingestion_ts"
            ),
        )
    )
    return df_products_unified

In [0]:
def unify_orders_data(df_offline_orders, df_online_orders):
    orders_transformations = [
        (trimming, {}),
    ]

    df_online_transformed = apply_transformations(
        df_online_orders, orders_transformations
    )

    df_online_orders_final = df_online_transformed.select(
        F.col("order_id").cast("string").alias("order_id"),
        F.col("order_ts").cast("timestamp").alias("order_ts"),
        F.col("customer_id").cast("string").alias("customer_id"),
        F.col("sku_id").cast("string").alias("sku_id"),
        F.col("qty").cast("int").alias("qty"),
        F.col("list_price").cast("double").alias("list_price"),
        F.col("discount").cast("double").alias("discount"),
        F.col("payment_mode").cast("string").alias("payment_mode"),
        F.lit("ONLINE").alias("store_num"),
        F.col("file_date"),
        F.col("source").cast("string").alias("source_system"),
        F.col("ingestion_ts"),
    )

    df_offline_transformed = apply_transformations(
        df_offline_orders, orders_transformations
    )

    df_offline_orders_final = df_offline_transformed.select(
        F.col("txn_id").cast("string").alias("order_id"),
        F.col("txn_time").cast("timestamp").alias("order_ts"),
        F.col("customer_id").cast("string").alias("customer_id"),
        F.col("sku").cast("string").alias("sku_id"),
        F.col("quantity").cast("int").alias("qty"),
        F.col("mrp").cast("double").alias("list_price"),
        F.col("discount").cast("double").alias("discount"),
        F.col("payment_mode").cast("string").alias("payment_mode"),
        F.col("store_num").cast("string").alias("store_num"),
        F.col("file_date"),
        F.col("source").cast("string").alias("source_system"),
        F.col("ingestion_ts"),
    )

    df_orders_unified = df_online_orders_final.union(df_offline_orders_final)
    return df_orders_unified

In [0]:
def cleanup_old_data(files, file_run):

    for file in files:
        print(f"Cleaning up {file} data for {file_run}")
        spark.sql(
            f"delete from retail_data.silver.{file}_unified where file_date = '{file_run}'"
        )

In [0]:
yest_date = datetime.now() - timedelta(days=1)
yest_date = yest_date.strftime("%Y%m%d")
dbutils.widgets.text("file_run", yest_date, "File Run Parameter")
file_run = dbutils.widgets.get("file_run")

In [0]:
sources = ["online", "offline"]
files = ["customers", "orders", "products"]

tables = {}
for source in sources:
    for file in files:
        tables[f"df_{source}_{file}"] = f"retail_data.bronze.{source}_{file}"

dfs = {
    k: spark.table(v).filter(F.col("file_date") == file_run) for k, v in tables.items()
}

In [0]:
cleanup_old_data(files, file_run)

save_unified_data(
    unify_customers_data,
    dfs["df_offline_customers"],
    dfs["df_online_customers"],
    "retail_data.silver.customers_unified",
    mode="append",
)

save_unified_data(
    unify_products_data,
    dfs["df_offline_products"],
    dfs["df_online_products"],
    "retail_data.silver.products_unified",
    mode="append",
)

save_unified_data(
    unify_orders_data,
    dfs["df_offline_orders"],
    dfs["df_online_orders"],
    "retail_data.silver.orders_unified",
    mode="append",
)